# Introducción al Aprendizaje Automático

## Optimización de hiperparámetros

### Conjunto de datos

Vamos a usar el mismo conjunto de datos que para la guía anterior, el MNIST.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.cm as cmap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

def plot_digit(data, subplot=plt):
    image = (data.reshape(28, 28)*255).astype(int)
    subplot.imshow(image, cmap = cmap.binary,
               interpolation="nearest")
    subplot.axis("off")

In [2]:
X, y = fetch_openml(
    "mnist_784", version=1, return_X_y=True, as_frame=False)

**Ejercicio**: de igual manera que la guía anterior, llevar a un problema binario, donde se distinga entre los números 3 y 8. Luego, normalizar el dataset.

In [3]:
# COMPLETAR

**Ejercicio:** hacer un train/test split del 80/20.

In [4]:
# COMPLETAR

**Ejercicio**: obtenga el nivel óptimo de regularización (`l1`) para un modelo de regresión logística, utilizando como métrica AUC-ROC o exactitud. Utilice validación cruzada con 5 folds. Grafique de manera conveniente.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


C_s = np.logspace(-3, 3, 10)

# COMPLETAR

In [ ]:
# COMPLETAR

**Ejercicio:** Entrene un modelo con el nivel de óptimo de regularización encontrado en el ejercicio anterior, y evalúe ese modelo en el conjunto de evaluación (test). ¿Cómo se compara con lo obtenido con validación cruzada?

In [ ]:
# COMPLETAR

**Ejercicio:** repita pero comparando distintos niveles de regularización y distintos tipos de regularización. Puedes hacerlo manualmente pero también explora la función `GridSearchCV` de Scikit-Learn. Lee atentamente su documentación.


In [14]:
from sklearn.model_selection import GridSearchCV

# COMPLETAR

**Ejercicio:** así como hicimos anteriormente, entrena el modelo óptimo obtenido previamente y evalualo en el conjunto de evaluación.


**Ejercicio:** optimiza los hiperparámetros de un árbol de decisión sobre este conjunto de datos. ¿Cuáles hiperparámetros explorarías? Además de `GridSearchCV` utiliza `RandomizedSearchCV`. Interpreta los resultados.


In [ ]:
# COMPLETAR

**Ejercicio:** ¿por qué es "obligatorio" usar validación cruzada? Haz el siguiente ejercicio para convencerte.

En algunos problemas financieros - por ejemplo, predecir si una acción va a subir o bajar en los próximos días - los desempeños suelen estar muy cerca del 0.5 para datasets balanceados (refiriéndonos a exactitud en este caso). Desempeños del 0.6 ya son considerados muy buenos, por no decir inalcanzables en muchísimas situaciones.

A continuación, vamos a cargar un dataset que contiene 1200 instancias, 5 atributos y una variable a predecir.

In [16]:
df = pd.read_csv('IAA_Guia_9_data.csv')
X = df.drop('y', axis=1)
y = df.y

1. Separar en conjuntos de entrenamiento, validación y evaluación. Cada uno con 850, 150 y 250 instancias, respectivamente.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=250, 
                                                    random_state=2023)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, 
                                                    test_size=150, 
                                                    random_state=2023)

print(X_train.shape, X_val.shape, X_test.shape)

2. Optimizar hiperparámetros para un Árbol de decisión usando entranamiento (*train*) y validación (*val*). Evaluar el desempeño del mejor modelo en evaluación (*test*).

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

Definimos los hiperparámetros a optimizar y sus posibles valores.

In [22]:
max_depth_list = [1,3,5,7,9,11]
min_samples_split_list = [10,20,30,40,50,60]
min_samples_leaf_list = [10,20,30,40,50,60]
max_features_list = [1,2,3,4,5]

Y entrenamos sobre todas las combinaciones.

In [23]:
np.random.seed(2023)
df_results = pd.DataFrame()
counter = 0

### Recorremos todas las combinaciones. Un loop por cada una.
for max_depth in max_depth_list:
    for COMPLETAR:
        for COMPLETAR:
            for COMPLETAR:
                tree = DecisionTreeClassifier(COMPLETAR)
                tree.COMPLETAR

                # Predecimos sobre nuestro set de entrenamieto
                y_train_pred = COMPLETAR

                # Predecimos sobre nuestro set de validacion
                y_val_pred = COMPLETAR
                
                # Guardamos resultados
                df_results.loc[counter, 'max_depth'] = max_depth
                df_results.loc[counter, 'min_samples_split'] = min_samples_split
                df_results.loc[counter, 'min_samples_leaf'] = min_samples_leaf
                df_results.loc[counter, 'max_features'] = max_features
                df_results.loc[counter, 'accuracy_train'] = COMPLETAR
                df_results.loc[counter, 'accuracy_val'] = COMPLETAR
                counter +=1

Veamos cuál es el mejor modelo para nuestro dataset.

In [ ]:
df_results.sort_values('accuracy_val', ascending=False).head()

Debería dar algo así como:

```
max_depth            11.000000
min_samples_split    30.000000
min_samples_leaf     30.000000
max_features          4.000000
accuracy_train        0.640000
accuracy_val          0.593333
```


Si te dio así, ¡un éxito! El modelo no parece estar sobreajustado. Además, el desempeño está bastante por encima del desempeño que asumiamos a priori. Si fuera un problema financiero de verdad, estás listo para hacerte rico.

4. EValúa su desempeño en `test`:

In [ ]:
y_test_pred = COMPLETAR
COMPLETAR

Probablemente te llevaste una desilusión. ¿Qué ocurrió?

Fíjense que nuestro conjunto de validación tiene 150 instancias, y nosotros probamos 1080 modelos. La probabilidad de que alguno de ellos pareciera andar bien por mera casualidad es altísima. ¿Cómo podríamos hacer para evitarnos este problema? La respuesta a esta pregunta - y a todas las situaciones planteadas previamente - es bastante intuitiva si la piensas, Validación Cruzada. 

**Ejercicio:** repetir el ejercicio anterior, pero haciendo GridSearchCV sobre el conjunto de entrenamiento. Luego, entrena un modelo con los mejores hiperparámetros y evalúa su desempeño en evaluación.

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=250, 
                                                    random_state=2023)

In [ ]:

max_depth_list = [1,3,5,7,9,11]
min_samples_split_list = [10,20,30,40,50,60]
min_samples_leaf_list = [10,20,30,40,50,60]
max_features_list = [1,2,3,4,5]
param_grid = {COMPLETAR
# COMPLETAR

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
# COMPLETAR